# Enterprise Text-to-SQL Agent MVP

This notebook is a reproducible walkthrough of the assessment prototype. It verifies the local Text-to-SQL pipeline, demonstrates schema retrieval and SQL safety controls, and summarizes the completed gold, Gemini, and local LLM evaluation outputs stored under `evaluation/results/`.

The live LLM call is patched in the workflow demo so the notebook can be executed repeatedly without API quota, local Ollama availability, or network dependency.

## 1. Setup

Import the compatibility module used by the Streamlit app and tests, then define the project paths used by the notebook.

In [1]:
from __future__ import annotations

import json
import sqlite3
from pathlib import Path
from unittest.mock import patch

import pandas as pd

import text_to_sql_agent_mvp as agent

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
RESULTS_DIR = ROOT / "evaluation" / "results"

pd.set_option("display.max_colwidth", 120)
print(f"Project root: {ROOT}")
print(f"Default provider/model: {agent.DEFAULT_PROVIDER} / {agent.DEFAULT_MODEL_NAME}")

Project root: /Users/tuanm.nguyen/Documents/aipa-text-to-sql-agent
Default provider/model: gemini / gemini-2.5-flash


## 2. Demo Databases

The MVP ships with three small SQLite databases that represent different enterprise-style domains: university, retail, and healthcare.

In [2]:
demo_dbs = {
    "university": DATA_DIR / "university_agent.db",
    "retail": DATA_DIR / "retail_analytics.db",
    "healthcare": DATA_DIR / "healthcare_analytics.db",
}

rows = []
for name, db_path in demo_dbs.items():
    if not db_path.exists():
        rows.append({"dataset": name, "path": str(db_path), "tables": "missing", "size_kb": None})
        continue
    with sqlite3.connect(db_path) as conn:
        table_count = conn.execute(
            "SELECT COUNT(*) FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'"
        ).fetchone()[0]
    rows.append({
        "dataset": name,
        "path": str(db_path.relative_to(ROOT)),
        "tables": table_count,
        "size_kb": round(db_path.stat().st_size / 1024, 1),
    })

pd.DataFrame(rows)

,dataset,path,tables,size_kb
0,university,data/university_agent.db,3,32.0
1,retail,data/retail_analytics.db,11,124.0
2,healthcare,data/healthcare_analytics.db,6,36.0


## 3. Schema Knowledge Representation

The agent treats database metadata as the knowledge source. It extracts table DDL, columns, constraints, relationships, and value hints before prompting the model.

In [3]:
schema_text = agent.get_schema(str(demo_dbs["university"]))
print(schema_text[:1400])

CREATE TABLE courses (
                course_id INTEGER PRIMARY KEY,
                course_code TEXT NOT NULL UNIQUE,
                course_name TEXT NOT NULL,
                credits INTEGER NOT NULL
            );

CREATE TABLE grades (
                student_id INTEGER NOT NULL,
                course_id INTEGER NOT NULL,
                semester TEXT NOT NULL,
                grade TEXT NOT NULL,
                score INTEGER NOT NULL,
                PRIMARY KEY (student_id, course_id, semester),
                FOREIGN KEY (student_id) REFERENCES students(student_id),
                FOREIGN KEY (course_id) REFERENCES courses(course_id)
            );

CREATE TABLE students (
                student_id INTEGER PRIMARY KEY,
                full_name TEXT NOT NULL,
                email TEXT NOT NULL UNIQUE,
                enrollment_year INTEGER NOT NULL,
                major TEXT NOT NULL
            );


## 4. Schema RAG Retrieval

Schema RAG retrieves the most relevant schema chunks for a user question. This reduces prompt size while keeping the expected tables available for SQL generation.

In [4]:
question = "Which product categories generated the most revenue?"
rag_context = agent.retrieve_schema_context(str(demo_dbs["retail"]), question, top_k=4)

retrieved = pd.DataFrame([
    {
        "rank": i + 1,
        "table": chunk.table_name,
        "score": round(chunk.score, 2),
        "matched_terms": ", ".join(chunk.matched_terms),
        "why": "; ".join(chunk.match_reasons[:3]),
    }
    for i, chunk in enumerate(rag_context.chunks)
])

print(f"Question: {question}")
print(f"Retrieval strategy: {rag_context.strategy}")
print(f"Schema chars: {rag_context.retrieved_schema_chars}/{rag_context.full_schema_chars}")
print(f"Prompt saved: {rag_context.prompt_savings_pct}%")
retrieved

Question: Which product categories generated the most revenue?
Retrieval strategy: hybrid-bm25-embedding-semantic-values-graph
Schema chars: 3528/5517
Prompt saved: 36.1%


,rank,table,score,matched_terms,why
0,1,returns,11.98,amount,column match: amount; semantic similarity: 0.16; embedding similarity: 0.29
1,2,products,9.68,product,column match: product; semantic similarity: 0.26; embedding similarity: 0.30
2,3,order_items,9.60,product,column match: product; semantic similarity: 0.21; embedding similarity: 0.27
3,4,inventory_snapshots,9.37,product,column match: product; semantic similarity: 0.18; embedding similarity: 0.23
4,5,suppliers,3.39,,semantic similarity: 0.09; embedding similarity: 0.32; foreign-key neighbor of products
5,6,orders,3.36,,semantic similarity: 0.12; embedding similarity: 0.22; foreign-key neighbor of order_items
6,7,stores,3.28,,semantic similarity: 0.11; embedding similarity: 0.25; foreign-key neighbor of inventory_snapshots


## 5. SQL Safety Controls

Generated SQL is validated before execution. Only read-only `SELECT` style queries are allowed, and metadata or write operations are blocked.

In [5]:
safety_examples = [
    "SELECT full_name FROM students LIMIT 5;",
    "DROP TABLE students;",
    "SELECT * FROM sqlite_master;",
    "UPDATE students SET major = 'CS';",
]

pd.DataFrame([
    {"sql": sql, "safe": agent.is_safe_query(sql)}
    for sql in safety_examples
])

,sql,safe
0,SELECT full_name FROM students LIMIT 5;,True
1,DROP TABLE students;,False
2,SELECT * FROM sqlite_master;,False
3,UPDATE students SET major = 'CS';,False


## 6. End-to-End Query Flow

This cell exercises the same public API used by the app. The LLM call is patched with known-good SQL so execution remains deterministic.

In [6]:
patched_sql = """
SELECT major, COUNT(*) AS student_count
FROM students
GROUP BY major
ORDER BY student_count DESC, major;
""".strip()

with patch.object(agent, "generate_sql", return_value=patched_sql):
    generated_sql, result = agent.ask_database_with_sql(
        "How many students are in each major?",
        db_path=str(demo_dbs["university"]),
        provider="gemini",
        model_name="gemini-2.5-flash",
        use_rag=True,
    )

print(generated_sql)
print(f"Error: {result.error}")
pd.DataFrame(result.rows, columns=result.columns)

SELECT major, COUNT(*) AS student_count
FROM students
GROUP BY major
ORDER BY student_count DESC, major;
Error: None


,major,student_count
0,Business,4
1,CS,4
2,DS,4
3,IT,4
4,Math,4


## 7. Benchmark Cases

The evaluation set contains 12 natural-language questions across the three datasets. Each case includes gold SQL and expected tables for result comparison and schema-retrieval checks.

In [7]:
cases = json.loads((ROOT / "evaluation" / "cases.json").read_text(encoding="utf-8"))
case_frame = pd.DataFrame(cases)
case_frame[["id", "dataset", "difficulty", "question", "expected_tables"]]

,id,dataset,difficulty,question,expected_tables
0,university_major_counts,university,easy,How many students are enrolled in each major?,[students]
1,university_average_score_by_course,university,medium,What is the average score for each course?,"[grades, courses]"
2,university_top_students_by_average_score,university,medium,Which five students have the highest average score?,"[students, grades]"
3,university_grade_distribution,university,easy,Show the number of grades awarded for each grade band.,[grades]
4,retail_revenue_by_region,retail,hard,Show total completed sales revenue by customer region.,"[orders, order_items, customers, regions]"
5,retail_revenue_by_category,retail,hard,Which product categories generate the most completed order revenue?,"[orders, order_items, products]"
6,retail_support_satisfaction_by_priority,retail,medium,What is the average customer support satisfaction score by ticket priority?,[customer_support_tickets]
7,retail_return_reason_counts,retail,easy,Which return reasons occur most often?,[returns]
8,healthcare_appointments_by_status,healthcare,easy,How many appointments are there for each status?,[appointments]
9,healthcare_treatment_cost_by_city,healthcare,hard,What is the average treatment cost by hospital city?,"[treatments, appointments, doctors, hospitals]"


## 8. Completed Evaluation Results

The repository now includes the gold SQL baseline, a Gemini 2.5 Flash run, and two local Ollama model runs. The table below is loaded from the committed result CSV files.

In [8]:
result_files = {
    "Gold SQL baseline": RESULTS_DIR / "evaluation_gold.csv",
    "Gemini 2.5 Flash": RESULTS_DIR / "evaluation_llm_gemini_gemini_2_5_flash.csv",
    "Ollama llama3:latest": RESULTS_DIR / "evaluation_llm_ollama_llama3_latest.csv",
    "Ollama gemma4:latest": RESULTS_DIR / "evaluation_llm_ollama_gemma4_latest.csv",
}

summary_rows = []
for label, path in result_files.items():
    df = pd.read_csv(path)
    total = len(df)
    summary_rows.append({
        "run": label,
        "cases": total,
        "safe_sql": f"{int(df['safe_sql'].sum())}/{total}",
        "executed": f"{int(df['execution_ok'].sum())}/{total}",
        "value_match": f"{int(df['value_match'].sum())}/{total}",
        "row_match": f"{int(df['row_match'].sum())}/{total}",
        "exact_match": f"{int(df['exact_result_match'].sum())}/{total}",
        "avg_schema_recall": round(df['schema_table_recall'].mean(), 3),
        "avg_prompt_saved_pct": round(df['prompt_saved_pct'].mean(), 1),
        "avg_latency_ms": round(df['latency_ms'].mean(), 2),
    })

summary = pd.DataFrame(summary_rows)
summary

,run,cases,safe_sql,executed,value_match,row_match,exact_match,avg_schema_recall,avg_prompt_saved_pct,avg_latency_ms
0,Gold SQL baseline,12,12/12,12/12,12/12,12/12,12/12,1.0,6.0,0.98
1,Gemini 2.5 Flash,12,12/12,12/12,11/12,6/12,0/12,1.0,6.0,3277.83
2,Ollama llama3:latest,12,12/12,12/12,8/12,5/12,0/12,1.0,6.0,3782.75
3,Ollama gemma4:latest,12,10/12,10/12,8/12,3/12,0/12,1.0,6.0,5651.83


### Result Interpretation

The gold SQL baseline confirms that all 12 benchmark questions and databases are valid. Gemini completed all cases safely and achieved the strongest LLM value match score. The local LLMs were fully offline but showed lower result fidelity, especially on harder retail joins. Gemma4 also produced two unsafe retail queries that were blocked by the safety layer.

In [9]:
metric_rows = []
for label, path in result_files.items():
    if label == "Gold SQL baseline":
        continue
    df = pd.read_csv(path)
    total = len(df)
    metric_rows.extend([
        {"run": label, "metric": "value_match", "rate": df["value_match"].sum() / total},
        {"run": label, "metric": "row_match", "rate": df["row_match"].sum() / total},
        {"run": label, "metric": "safe_sql", "rate": df["safe_sql"].sum() / total},
    ])

rates = pd.DataFrame(metric_rows)
rates["rate_pct"] = (rates["rate"] * 100).round(1)
rates.pivot(index="run", columns="metric", values="rate_pct").reset_index()

metric,run,row_match,safe_sql,value_match
0,Gemini 2.5 Flash,50.0,100.0,91.7
1,Ollama gemma4:latest,25.0,83.3,66.7
2,Ollama llama3:latest,41.7,100.0,66.7


## 9. Re-run Commands

Use these terminal commands when you need to regenerate the result files. The notebook itself does not execute them, because hosted model quota and local Ollama model availability are environment-dependent.

In [10]:
commands = [
    "python3 scripts/evaluate_text_to_sql.py --mode gold",
    "python3 scripts/evaluate_text_to_sql.py --mode llm --provider gemini --model gemini-2.5-flash --max-cases 12 --max-retries 1 --retry-base-seconds 5",
    "python3 scripts/evaluate_text_to_sql.py --mode llm --provider ollama --model llama3:latest --resume",
    "python3 scripts/evaluate_text_to_sql.py --mode llm --provider ollama --model gemma4:latest --resume",
]
print("\n".join(commands))

python3 scripts/evaluate_text_to_sql.py --mode gold
python3 scripts/evaluate_text_to_sql.py --mode llm --provider gemini --model gemini-2.5-flash --max-cases 12 --max-retries 1 --retry-base-seconds 5
python3 scripts/evaluate_text_to_sql.py --mode llm --provider ollama --model llama3:latest --resume
python3 scripts/evaluate_text_to_sql.py --mode llm --provider ollama --model gemma4:latest --resume


## 10. Gemini Quota Notes

The final Gemini run used the multi-key manager and completed the 12-case benchmark without quota failures. Detailed notes are stored in `evaluation/results/gemini_12_case_quota_notes.md`.

In [11]:
quota_notes = (RESULTS_DIR / "gemini_12_case_quota_notes.md").read_text(encoding="utf-8")
print("\n".join(quota_notes.splitlines()[:18]))

# Gemini 12-Case Evaluation Notes

Run command:

```bash
python3 scripts/evaluate_text_to_sql.py --mode llm --provider gemini --max-cases 12 --max-retries 1 --retry-base-seconds 5
```

Output files:

- `evaluation/results/evaluation_llm_gemini_gemini_2_5_flash.csv`
- `evaluation/results/evaluation_llm_gemini_gemini_2_5_flash.md`

## Key Rotation

At the time of this run, `.env` contained one configured Gemini key source:

- `GOOGLE_API_KEY`


## 11. Assessment Takeaways

- The prototype satisfies the required Text-to-SQL flow: natural language input, schema-aware prompting, SQL generation, validation, execution, and tabular answer display.
- Retrieval keeps all expected tables available in the benchmark while reducing prompt size where schemas are larger.
- The safety layer is necessary because LLM outputs can be executable but still unsafe or semantically wrong.
- Hosted Gemini gave the best LLM result fidelity in this run; local LLMs are attractive for privacy but need more prompt tuning or stronger models for complex joins.